# Phase 3 — Grading pathway (M1)

Trains the image-level grader and runs the **B1–B5** sweep that picks one recipe.
Validation only: no test set is touched in this phase, and no calibration is fitted.

## Notebook settings (right-hand panel)

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Persistence | **Files only** — lets a killed run resume instead of restart |
| Internet | **On** (git clone) |
| Environment | **Pin to original environment** |

T4 has tensor cores, so AMP roughly halves epoch time; P100 does not and is the wrong
choice here. The script trains on **one** GPU — the second T4 sits idle, which costs
nothing, because Kaggle bills session wall-clock rather than per-device.

## Inputs to add

`verify-dr-cache-512`, `verify-dr-idrid-masks`, and the Phase 2 manifests
(`verify-dr-manifests`, or `02_manifests.ipynb` as a notebook input). **The raw
datasets are no longer needed** — from here on everything reads the cache.

## The order, and why it matters

| Order | Experiment | Decides |
|---|---|---|
| 1 | B1 resolution 384 / 512 / 768 | Working resolution |
| 2 | B2 backbone B0 / ResNet50 | Encoder |
| 3 | B3 head CE / ordinal / focal-ordinal | Output head |
| 4 | B4 sampler | Class exposure |
| 5 | B5 eye-pair fusion | Whether fusion is in |

Each step keeps the winner of the one before, so B1 is run three times, then B2 twice
on B1's winner, and so on. That is ~10 runs, not 3×2×3×3×2.

> **Watch B1 on grade-1 recall, not QWK.** Grade 1 is microaneurysms only — 10–20 px
> at full resolution. A model that quietly never predicts grade 1 can still post a
> respectable QWK: on a realistic distribution that is 90% accuracy with a macro-F1 of
> 0.47. The verdict cell below applies this rule for you.

## Exit condition

One recipe chosen, all five results in `docs/04_experiment_register.md` with validation
numbers. **You have not touched a test set yet.**

## 1 · Clone the repo

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

## 2 · Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Set Accelerator to 'GPU T4 x2' in the right-hand panel, then "
        "re-run from the top.")

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory / 2**30:.1f} GB  sm_{p.major}{p.minor}")

if torch.cuda.get_device_properties(0).major < 7:
    print("\nNOTE: no tensor cores on this GPU, so AMP buys little.")
    print("      GPU T4 x2 is roughly twice as fast here and costs the same quota.")

## 3 · Helpers

In [ ]:
from pathlib import Path
from collections import Counter
import json, shlex, subprocess, sys, time, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    """Run a training job, streaming its output live.

    The earlier notebooks use capture_output=True, which is fine for a two-minute
    manifest build and useless here: you would see nothing at all until a
    three-hour job exited. Streaming means a per-epoch line appears as it happens,
    so a run that is going wrong can be stopped in epoch 1 rather than hour 3.
    """
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        if code == 2:
            print("\nExit 2 is an argument error. Usually the cloned scripts are stale:")
            print("re-run the clone cell at the top, then run from there.")
        raise RuntimeError(f"training failed with exit {code}")

def cache_roots():
    """Every directory that looks like a build_cache.py output root.

    build_cache.py writes <root>/<dataset>/cache_report.json, so a report file
    identifies its root two levels up. /kaggle/input is searched first: a stale
    copy in /kaggle/working must never silently win over the dataset you attached.
    """
    found = []
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for root, _ in Counter(r.parent.parent for r in base.rglob("cache_report.json")).most_common():
            found.append(root)
    return found

def resolve_datasets(roots):
    """dataset -> the root holding the best copy of it.

    The cache is legitimately split across published datasets: the full build
    plus a later top-up. IDRiD's lesion masks ship as their own dataset
    ('verify-dr-idrid-masks'), so a single root shows idrid with zero mask
    channels even when the masks are attached. When a dataset appears in more
    than one root, the copy with more mask channels wins.
    """
    best = {}
    for root in roots:
        for d in sorted(x for x in root.iterdir() if x.is_dir()):
            if not (d / "cache_report.json").exists():
                continue
            masks = d / "masks"
            score = len(list(masks.iterdir())) if masks.is_dir() else 0
            if d.name not in best or score > best[d.name][1]:
                best[d.name] = (root, score)
    return {name: root for name, (root, _) in best.items()}

def extract_cache(dest=WORK / "cache512"):
    """Extract a cache published as a zip. Idempotent within a session.

    This costs GPU-session minutes, which come out of the 30 h/week quota. If you
    hit it every run, re-upload the cache to Kaggle as a *dataset* rather than as
    notebook output -- an uploaded zip is unpacked by Kaggle once, server-side.
    """
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
        except (zipfile.BadZipFile, OSError):
            continue
        if not any(n.endswith("cache_report.json") for n in names):
            continue
        marker = dest / ".extracted_from"
        if marker.exists() and marker.read_text().strip() == z.name:
            print(f"already extracted from {z.name}")
            return dest
        print(f"extracting {z.name} ({z.stat().st_size / 2**30:.1f} GB) -> {dest}")
        dest.mkdir(parents=True, exist_ok=True)
        started = time.time()
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        marker.write_text(z.name)
        print(f"extracted in {(time.time() - started) / 60:.1f} min")
        return dest
    return None

def find_manifest_dir():
    """Phase 2's output: the directory holding dataset_plan.json and the variants."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        hits = sorted(base.rglob("dataset_plan.json"))
        if hits:
            return hits[0].parent
    return None

## 4 · Find the cache and the manifests

In [ ]:
ROOTS = cache_roots()
if not ROOTS and extract_cache():
    ROOTS = cache_roots()
MANIFEST_DIR = find_manifest_dir()

if not ROOTS:
    raise RuntimeError(
        "No cache found. Attach verify-dr-cache-512 (and verify-dr-idrid-masks) "
        "under Add Data in the right-hand panel.")
if MANIFEST_DIR is None:
    raise RuntimeError(
        "No manifests found. Attach the Phase 2 output (verify-dr-manifests), or "
        "add 02_manifests.ipynb as a notebook input.")

DATASETS = resolve_datasets(ROOTS)
print("cache roots  :")
for r in ROOTS:
    print("   ", r)
print("manifest dir :", MANIFEST_DIR)

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

print("\ndatasets in the cache:")
print(f"  {'dataset':<12}{'images':>8}  {'masks':>5}  root")
for name, root in sorted(DATASETS.items()):
    d = root / name
    images = d / 'images'
    # rglob, not glob: glob('*') counts direct children only, so a nested
    # layout reports 1 and looks like catastrophic data loss when nothing
    # is actually wrong.
    n_img = sum(1 for f in images.rglob('*') if f.suffix.lower() in IMAGE_SUFFIXES) \
        if images.is_dir() else 0
    n_msk = len([m for m in (d / 'masks').iterdir() if m.is_dir()]) \
        if (d / 'masks').is_dir() else 0
    print(f"  {name:<12}{n_img:>8}  {n_msk:>5}  {root}")

    if images.is_dir():
        subdirs = [x for x in images.glob('*') if x.is_dir()]
        if subdirs and n_img:
            print(f"               ^ nested under {len(subdirs)} subdirectorie(s), "
                  f"e.g. {subdirs[0].name}/ - fine, the manifest stores full paths")
    if n_img == 0:
        print(f"               ^ NO IMAGES - {name} is empty in every attached root")

print("\nmanifests available:")
for c in sorted(MANIFEST_DIR.glob("*.csv")):
    print("  ", c.name)

print("\nMasks are a Phase 4 concern. M1 grades whole images and reads images +")
print("grades only, so 0 mask channels here blocks nothing in Phase 3. What")
print("matters now is that the datasets your chosen manifest names have images.")

# Every root goes to --cache-root, so each dataset resolves to the root that
# actually holds it instead of all of them being forced onto one.
CACHE_FLAGS = ' '.join(q(r) for r in ROOTS)

## 5 · Choose the experiment

Edit this cell, then run the rest of the notebook. Everything below reads from it.

In [ ]:
# ========================== EDIT THIS CELL ===============================
# One experiment per run (notebooks/README.md). A crash then costs one result,
# not a batch. Run B1 -> B5 in order; each one's winner feeds the next.

EXPERIMENT = "B1_res512"

ARGS = dict(
    manifest        = "eyepacs_balanced_1000.csv",
    image_size      = 512,
    backbone        = "efficientnet_b0",
    head            = "ordinal_focal",
    sampler         = "stratified_exposure",
    batch_size      = 24,
    epochs          = 10,
    lr              = 3e-4,
    epoch_samples   = None,        # None -> one draw per training row
    eye_pair_fusion = False,
)

RESUME = True   # Safe to leave on. The script refuses to resume across a change
                # of backbone, head, image size or fusion, so a re-run after
                # editing this cell starts clean instead of silently blending two
                # architectures into one checkpoint.
# =========================================================================

# Batch sizes that fit a 16 GB T4 with AMP. Drop one step if you hit OOM.
#
#   resolution   efficientnet_b0   resnet50
#      384             48             24
#      512             24             12
#      768             10              6
#
# Halve these again when eye_pair_fusion is on: it encodes two images per sample.

## 6 · Smoke test first

In [ ]:
# Two minutes now against three hours of a run that was never going to work.
# Same code path, 300 rows, one epoch: catches a bad manifest, a missing cache,
# an OOM batch size, a typo in a flag.
smoke = [
    f"python {q(REPO_DIR / 'scripts/train_grading.py')}",
    f"--manifest {q(MANIFEST_DIR / ARGS['manifest'])}",
    f"--experiment SMOKE_{EXPERIMENT}",
    f"--results-dir {q(WORK / 'smoke')}",
    f"--cache-root {CACHE_FLAGS}",
    f"--image-size {ARGS['image_size']}",
    f"--backbone {ARGS['backbone']}",
    f"--head {ARGS['head']}",
    f"--batch-size {ARGS['batch_size']}",
    "--epochs 1 --warmup-epochs 1 --limit 300 --workers 2",
]
if ARGS["eye_pair_fusion"]:
    smoke.append("--eye-pair-fusion")
run(" ".join(smoke))
print("\nSMOKE TEST PASSED - the real run below uses the same code path.")

## 7 · Train

A per-epoch line appears as it happens. A checkpoint is written **every epoch**, so if
Kaggle kills the session at ~12 h you lose one epoch rather than the run — re-attach
the notebook output and run again with `RESUME = True`.

In [ ]:
flags = [
    f"python {q(REPO_DIR / 'scripts/train_grading.py')}",
    f"--manifest {q(MANIFEST_DIR / ARGS['manifest'])}",
    f"--experiment {q(EXPERIMENT)}",
    f"--results-dir {q(RESULTS)}",
    f"--cache-root {CACHE_FLAGS}",
    f"--image-size {ARGS['image_size']}",
    f"--backbone {ARGS['backbone']}",
    f"--head {ARGS['head']}",
    f"--sampler {ARGS['sampler']}",
    f"--batch-size {ARGS['batch_size']}",
    f"--epochs {ARGS['epochs']}",
    f"--lr {ARGS['lr']}",
    "--workers 2",
]
if ARGS["epoch_samples"]:
    flags.append(f"--epoch-samples {ARGS['epoch_samples']}")
if ARGS["eye_pair_fusion"]:
    flags.append("--eye-pair-fusion")
if RESUME:
    flags.append("--resume")

started = time.time()
run(" ".join(flags))
print(f"\nwall clock: {(time.time() - started) / 3600:.2f} GPU-hours")

## 8 · Result, and the B1 verdict

In [ ]:
m = json.loads((RESULTS / EXPERIMENT / "metrics.json").read_text())
best = m["best_val"]

print(f"{m['experiment']}   best epoch {m['best_epoch']} of {m['epochs_run']}   "
      f"{m['minutes']:.0f} min")
print(f"  QWK       {best['qwk']:.4f}")
print(f"  macro-F1  {best['macro_f1']:.4f}")
print(f"  MAE       {best['mae']:.4f}")
print(f"  ECE       {best['ece']:.4f}   (uncalibrated - Phase 7 fixes this)")
print(f"  AUROC rDR {best.get('auroc_rdr', float('nan')):.4f}   "
      f"VTDR {best.get('auroc_vtdr', float('nan')):.4f}")

print("\n  grade      n   recall  precision      F1")
for g in map(str, range(5)):
    r, p, f = (best["per_class_recall"][g], best["per_class_precision"][g],
               best["per_class_f1"][g])
    fmt = lambda v: "   -  " if v != v else f"{v:.3f} "
    bar = "#" * int(30 * (f if f == f else 0))
    print(f"    {g}  {best['support'][g]:>6}   {fmt(r)}   {fmt(p)}   {fmt(f)} {bar}")

print("\n  confusion (rows = true, cols = predicted)")
for i, row in enumerate(best["confusion"]):
    print(f"    {i} | " + " ".join(f"{v:>6}" for v in row))

# ---- the B1 decision rule, applied rather than described -------------------
# Grade-1 RECALL alone cannot decide this. A model that predicts grade 1 for
# every image scores 1.000 recall on grade 1 and is worthless, so collapse is
# checked first and the grade itself is judged on F1.
print("\n" + "=" * 72)
distinct = best.get("distinct_predictions")
print(f"distinct grades predicted: {distinct} of 5")
r1, p1, f1 = (best["per_class_recall"]["1"], best["per_class_precision"]["1"],
              best["per_class_f1"]["1"])

if distinct == 1:
    only = best["confusion"][0].index(max(best["confusion"][0]))
    print(f"COLLAPSED: the model predicts grade {only} for every image.")
    print("Per-class recall is unreadable in this state - the one predicted grade")
    print("scores 1.000 by construction. Nothing here decides B1.")
    print("Train longer, or raise the learning rate: this is an undertrained model,")
    print("not a resolution verdict.")
elif r1 != r1:
    print("Grade 1 is absent from validation - this manifest cannot decide B1.")
elif f1 < 0.05:
    print(f"GRADE-1 F1 {f1:.3f}  (recall {r1:.3f}, precision {p1:.3f})")
    print("The model has effectively learned to skip the class. A QWK of any size")
    print("on top of this is hollow - skipping grade 1 entirely still scores ~0.97")
    print("QWK on a realistic distribution, because grade 1 sits one step from")
    print("grade 0 and quadratic weights barely punish it.")
    print("Grade 1 is microaneurysms only, 10-20 px at full resolution, and they")
    print("vanish under downsampling. Try higher before concluding anything - and")
    print("if grade-1 F1 is near zero at EVERY resolution, say so in the thesis.")
    print("That is a finding, not a failure to hide.")
elif f1 < 0.20:
    print(f"Grade-1 F1 {f1:.3f} is weak but not collapsed "
          f"(recall {r1:.3f}, precision {p1:.3f}).")
    print("Compare against the other resolutions before choosing - B1 is decided")
    print("on this number, not on QWK.")
else:
    print(f"Grade-1 F1 {f1:.3f} (recall {r1:.3f}, precision {p1:.3f}) - the class is")
    print("being learned rather than skipped.")
    if p1 < 0.5 * r1:
        print("Precision is well below recall, so grade 1 is being over-predicted:")
        print("much of what it labels grade 1 is not. Read the confusion matrix rows.")

if distinct is not None and 1 < distinct < 4:
    print(f"\nNote: only {distinct} of the 5 grades are ever predicted. That is not a")
    print("collapse, but it is close to one, and QWK tolerates it better than the")
    print("clinic would. Weigh this against the other resolutions.")
print("=" * 72)

## 9 · Everything run so far

In [ ]:
import pandas as pd

rows = []
for mp in sorted(RESULTS.glob("*/metrics.json")):
    m = json.loads(mp.read_text())
    if not m.get("best_val"):
        continue
    b, c = m["best_val"], m["config"]
    rows.append({
        "experiment": m["experiment"],
        "res": c["image_size"], "backbone": c["backbone"].replace("efficientnet_", "eff_"),
        "head": c["head"], "sampler": c["sampler"][:10], "fusion": c["eye_pair_fusion"],
        "QWK": round(b["qwk"], 4), "macroF1": round(b["macro_f1"], 4),
        "g1_F1": round(b["per_class_f1"]["1"], 3) if b["per_class_f1"]["1"] == b["per_class_f1"]["1"] else None,
        "g1_rec": round(b["per_class_recall"]["1"], 3) if b["per_class_recall"]["1"] == b["per_class_recall"]["1"] else None,
        "n_pred": b.get("distinct_predictions"),
        "MAE": round(b["mae"], 3), "min": m["minutes"],
    })

if rows:
    table = pd.DataFrame(rows).sort_values("QWK", ascending=False)
    # n_pred of 1 means the run collapsed onto one grade; its QWK means nothing.
    print(table.to_string(index=False))
    table.to_csv(WORK / "stage_b_summary.csv", index=False)
    print("\nCopy these into docs/04_experiment_register.md before the session ends.")
    print("/kaggle/working does not survive, and an unrecorded run has to be paid for twice.")
else:
    print("No completed runs in", RESULTS)

---
## 10 · Save

**Save Version → Save & Run All (Commit).** Publish `/kaggle/working/results` as
**`verify-dr-stage-b`**, or attach this notebook's output to the next run so `RESUME`
can find the checkpoint.

Then, before you close the tab:

1. Copy the row from § 9 into `docs/04_experiment_register.md` — experiment id, the
   validation numbers, and the date.
2. Set that experiment's **Status** to done.
3. If anything deviated from the plan — a batch size dropped for OOM, an epoch count
   changed on resume — write it down. The script prints a `note:` line for each one.

`/kaggle/working` does not survive the session. A run whose number was never written
down has to be paid for twice, out of the same 30 GPU-h/week.

> **Do not fit calibration here, and do not look at a test set.** Phase 3 chooses a
> recipe on validation. Temperature scaling is Phase 7, on the calibration split, and
> APTOS and Messidor-2 stay locked until the pre-registration is committed.